# Lab 01: Exercise Solutions
## Reference answers for Lesson 1 — Particle in a 1D Box

These reference solutions let you verify your work on the
[Lesson 1 exercises](lesson-01.ipynb#Exercises). Each exercise has a
self-contained function that prints expected results and generates
diagnostic plots.

**How to use:** Try the exercise yourself first, then call the
corresponding function (e.g., `exercise_1_reference()`) to compare.

In [ ]:
import numpy as np
from scipy import linalg
import matplotlib.pyplot as plt
from quantum1d import QuantumSystem1D, STATE_COLORS, STATE_LINESTYLES

# Constants used across exercises
bohr_per_nm = 18.8973
ha_to_eV = 27.2114
L = 1.0 * bohr_per_nm  # 1 nm box in bohr

def V_box(x):
    return np.zeros_like(x)

print(f'Solver loaded. L = {L:.2f} bohr = 1.00 nm')

---
## Exercise 1: Energy scaling with box size

**Question:** How do the energies scale with $L$? Verify the $1/L^2$ dependence.

**Key result:** $E_1 \times L^2 = \pi^2/2 \approx 4.9348$ for all box sizes.

In [ ]:
def exercise_1_reference():
    L_values = np.array([0.5, 1.0, 1.5, 2.0, 3.0]) * bohr_per_nm
    E1_values = []

    for L_test in L_values:
        sys_test = QuantumSystem1D(0, L_test, 200, V_box)
        E_test, _ = sys_test.solve(n_states=1)
        E1_values.append(E_test[0])

    E1_values = np.array(E1_values)
    E1_analytical = np.pi**2 / (2 * L_values**2)

    print("Exercise 1: E₁ vs box size L")
    print("=" * 60)
    print(f"{'L (nm)':>8}  {'L (bohr)':>10}  {'E₁ (Ha)':>12}  {'π²/(2L²)':>12}  {'Ratio':>8}")
    print("-" * 60)
    for i, Lv in enumerate(L_values):
        print(f"{Lv/bohr_per_nm:>8.1f}  {Lv:>10.2f}  {E1_values[i]:>12.6f}  "
              f"{E1_analytical[i]:>12.6f}  {E1_values[i]/E1_analytical[i]:>8.5f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    L_nm_arr = L_values / bohr_per_nm

    ax1.plot(L_nm_arr, E1_values, 'o-', label='Numerical $E_1$')
    ax1.plot(L_nm_arr, E1_analytical, 'x--', label=r'$\pi^2/(2L^2)$')
    ax1.set_xlabel('L (nm)')
    ax1.set_ylabel('$E_1$ (hartree)')
    ax1.set_title('Ground state energy vs box size')
    ax1.legend()

    ax2.plot(L_nm_arr, E1_values * L_values**2, 'o-')
    ax2.axhline(y=np.pi**2 / 2, color='gray', linestyle='--',
                label=f'$\\pi^2/2 = {np.pi**2/2:.4f}$')
    ax2.set_xlabel('L (nm)')
    ax2.set_ylabel(r'$E_1 \times L^2$ (hartree·bohr²)')
    ax2.set_title('$E_1 L^2$ should be constant')
    ax2.legend()
    plt.tight_layout()
    plt.show()
    print(f"\nE₁ × L² values: {np.round(E1_values * L_values**2, 4)}")
    print(f"Expected: π²/2 = {np.pi**2/2:.4f}")

exercise_1_reference()

---
## Exercise 2: Heavier particle (proton)

**Question:** Change `mass` to 1836. How do the energies change? Why?

**Key result:** $E_n \propto 1/m$, so a proton (1836× heavier) has 1836× smaller energies.

In [ ]:
def exercise_2_reference():
    mass_proton = 1836.15267343  # proton mass in units of m_e

    sys_electron = QuantumSystem1D(0, L, 200, V_box, mass=1.0)
    E_electron, _ = sys_electron.solve(n_states=4)

    sys_proton = QuantumSystem1D(0, L, 200, V_box, mass=mass_proton)
    E_proton, _ = sys_proton.solve(n_states=4)

    print("Exercise 2: Electron vs proton in a 1 nm box")
    print("=" * 65)
    print(f"{'n':>3}  {'E_electron (Ha)':>16}  {'E_proton (Ha)':>16}  {'Ratio':>10}")
    print("-" * 65)
    for i in range(4):
        print(f"{i+1:>3}  {E_electron[i]:>16.8f}  {E_proton[i]:>16.8f}  "
              f"{E_electron[i]/E_proton[i]:>10.1f}")

    print(f"\nExpected ratio: m_proton/m_electron = {mass_proton:.1f}")
    print("Since E_n = n²π²/(2mL²), energies scale as 1/m.")
    print(f"The proton, being {mass_proton:.0f}× heavier, has {mass_proton:.0f}× smaller energies.")
    print(f"Proton ground state: {E_proton[0]*ha_to_eV*1000:.3f} meV "
          f"(electron: {E_electron[0]*ha_to_eV*1000:.1f} meV)")

exercise_2_reference()

---
## Exercise 3: Finite well

**Question:** How many bound states exist for a given $V_0$ and $w$?

**Key results:**
- Bound states have $E < V_0$ and exhibit exponential tails outside the well
- Number of bound states $\approx \lceil w\sqrt{2mV_0}/\pi \rceil$
- The wavefunctions penetrate into the classically forbidden region — tunneling!

In [ ]:
def exercise_3_reference():
    w = 10.0      # well width in bohr
    V0 = 0.1      # barrier height in hartree
    domain_half = 30.0

    def V_finite_well(x):
        V = np.full_like(x, V0)
        V[np.abs(x) < w / 2] = 0.0
        return V

    sys_fw = QuantumSystem1D(-domain_half, domain_half, 500, V_finite_well)
    E_fw, states_fw = sys_fw.solve(n_states=10)

    n_bound = int(np.sum(E_fw < V0))

    print(f"Exercise 3: Finite well (w = {w:.1f} bohr, V₀ = {V0:.3f} Ha = {V0*ha_to_eV:.2f} eV)")
    print("=" * 60)
    print(f"{'n':>3}  {'E (Ha)':>12}  {'E (eV)':>10}  {'Bound?':>8}")
    print("-" * 60)
    for i in range(min(8, len(E_fw))):
        bound = "yes" if E_fw[i] < V0 else "no"
        print(f"{i+1:>3}  {E_fw[i]:>12.6f}  {E_fw[i]*ha_to_eV:>10.4f}  {bound:>8}")
    print(f"\nNumber of bound states: {n_bound}")

    n_est = w * np.sqrt(2 * V0) / np.pi
    print(f"Analytical estimate: w√(2mV₀)/π = {n_est:.2f}  →  {int(np.ceil(n_est))} bound states")

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    x_full = np.concatenate([[sys_fw.x_min], sys_fw.x, [sys_fw.x_max]])
    V_plot = np.concatenate([[V0], V_finite_well(sys_fw.x), [V0]])

    ax1.fill_between(x_full, 0, V_plot, alpha=0.1, color='gray')
    ax1.plot(x_full, V_plot, 'k-', linewidth=0.8)
    for i in range(min(n_bound, 4)):
        phi = np.concatenate([[0], states_fw[:, i], [0]])
        color = STATE_COLORS[i % len(STATE_COLORS)]
        ls = STATE_LINESTYLES[i % len(STATE_LINESTYLES)]
        scale = 0.02
        ax1.plot(x_full, phi * scale + E_fw[i], color=color, linestyle=ls,
                 linewidth=1.2)
        ax1.axhline(y=E_fw[i], color=color, linewidth=0.5, linestyle=':', alpha=0.5)
    ax1.set_xlabel('x (bohr)')
    ax1.set_ylabel('Energy (hartree)')
    ax1.set_title('Bound states in finite well')
    ax1.set_xlim(-20, 20)

    for i in range(min(n_bound, 4)):
        phi = np.concatenate([[0], states_fw[:, i], [0]])
        color = STATE_COLORS[i % len(STATE_COLORS)]
        ls = STATE_LINESTYLES[i % len(STATE_LINESTYLES)]
        ax2.plot(x_full, phi, color=color, linestyle=ls, linewidth=1.5,
                 label=f'n={i+1}, E={E_fw[i]:.5f}')
    ax2.axvspan(-w/2, w/2, alpha=0.06, color='blue', label='Well region')
    ax2.set_xlabel('x (bohr)')
    ax2.set_ylabel(r'$\phi(x)$')
    ax2.set_title('Wavefunctions (note exponential tails)')
    ax2.legend(fontsize=7)
    ax2.set_xlim(-20, 20)
    plt.tight_layout()
    plt.show()
    print("\nKey observation: wavefunctions have exponential tails outside the well.")
    print("This is classically forbidden — the particle tunnels into the barrier.")

exercise_3_reference()

---
## Exercise 4: Harmonic oscillator

**Question:** Compare numerical energies to the exact result $E_n = \omega(n + 1/2)$.

**Key results:**
- Equally spaced energy levels (unlike the $n^2$ scaling in the box)
- Domain must be large enough for the wavefunctions to decay to zero at the boundaries
- Gaussian-shaped ground state, Hermite polynomial structure in excited states

In [ ]:
def exercise_4_reference():
    omega = 0.01  # angular frequency in atomic units

    # Domain: classical turning point for n=5 is sqrt(2*5.5*omega/omega²) ≈ 33 bohr
    x_range = 50.0

    def V_harmonic(x):
        return 0.5 * omega**2 * x**2

    sys_ho = QuantumSystem1D(-x_range, x_range, 500, V_harmonic)
    E_ho, states_ho = sys_ho.solve(n_states=6)

    print(f"Exercise 4: Harmonic oscillator (ω = {omega} a.u.)")
    print("=" * 65)
    print(f"{'n':>3}  {'E_numerical':>14}  {'E_exact':>14}  {'Rel. Error':>12}")
    print("-" * 65)
    for i in range(6):
        E_exact = omega * (i + 0.5)
        rel_err = abs(E_ho[i] - E_exact) / E_exact
        print(f"{i:>3}  {E_ho[i]:>14.8f}  {E_exact:>14.8f}  {rel_err:>12.2e}")

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    x_full = np.concatenate([[sys_ho.x_min], sys_ho.x, [sys_ho.x_max]])
    V_plot = np.concatenate([[V_harmonic(-x_range)], V_harmonic(sys_ho.x),
                              [V_harmonic(x_range)]])

    ax1.plot(x_full, V_plot, 'k-', linewidth=0.8, label=r'$V = \frac{1}{2}\omega^2 x^2$')
    for i in range(4):
        E = E_ho[i]
        phi = np.concatenate([[0], states_ho[:, i], [0]])
        color = STATE_COLORS[i % len(STATE_COLORS)]
        ls = STATE_LINESTYLES[i % len(STATE_LINESTYLES)]
        scale = 5.0
        ax1.plot(x_full, phi * scale + E, color=color, linestyle=ls, linewidth=1.2)
        ax1.axhline(y=E, color=color, linewidth=0.5, linestyle=':', alpha=0.5)
    ax1.set_xlim(-40, 40)
    ax1.set_ylim(-0.001, omega * 5)
    ax1.set_xlabel('x (bohr)')
    ax1.set_ylabel('Energy (hartree)')
    ax1.set_title('Harmonic oscillator eigenstates')

    n_vals = np.arange(6)
    ax2.plot(n_vals, E_ho, 'o', markersize=8, label='Numerical')
    ax2.plot(n_vals, omega * (n_vals + 0.5), 'x', markersize=10,
             label=r'$\omega(n + 1/2)$')
    ax2.set_xlabel('Quantum number n')
    ax2.set_ylabel('Energy (hartree)')
    ax2.set_title('Energy levels: numerical vs exact')
    ax2.legend()
    plt.tight_layout()
    plt.show()
    print(f"\nEqual spacing: ΔE = ω = {omega} Ha (exact).")
    print("Numerical spacings:", np.diff(E_ho).round(8))

exercise_4_reference()

---
## Exercise 5: 4th-order finite differences

**Question:** Implement a 4th-order scheme and demonstrate $O(\Delta x^4)$ convergence.

**Key result:** The 4th-order stencil $(-1, 16, -30, 16, -1)/(12\Delta x^2)$ converges
as $N^{-4}$ instead of $N^{-2}$, meaning far fewer grid points are needed for the
same accuracy.

In [ ]:
def exercise_5_reference():
    # 4th-order second derivative stencil:
    # f''(x) ≈ (-f_{j-2} + 16f_{j-1} - 30f_j + 16f_{j+1} - f_{j+2}) / (12 Δx²)

    def solve_4th_order(x_min, x_max, N, V_func, mass=1.0, n_states=4):
        dx = (x_max - x_min) / (N + 1)
        x = np.linspace(x_min + dx, x_max - dx, N)

        coeff = 1.0 / (2.0 * mass * 12.0 * dx**2)
        T = np.zeros((N, N))
        for j in range(N):
            T[j, j] = 30.0 * coeff
            if j > 0:   T[j, j-1] = -16.0 * coeff
            if j < N-1: T[j, j+1] = -16.0 * coeff
            if j > 1:   T[j, j-2] = 1.0 * coeff
            if j < N-2: T[j, j+2] = 1.0 * coeff

        V = np.diag(V_func(x))
        H = T + V
        eigenvalues, _ = linalg.eigh(H)
        return eigenvalues[:n_states]

    N_values = [20, 50, 100, 200, 500]
    E1_exact = np.pi**2 / (2 * L**2)

    errors_2nd = []
    errors_4th = []
    for N in N_values:
        sys2 = QuantumSystem1D(0, L, N, V_box)
        E2, _ = sys2.solve(n_states=1)
        errors_2nd.append(abs(E2[0] - E1_exact) / E1_exact)

        E4 = solve_4th_order(0, L, N, V_box, n_states=1)
        errors_4th.append(abs(E4[0] - E1_exact) / E1_exact)

    print("Exercise 5: Convergence — 2nd-order vs 4th-order finite differences")
    print("=" * 60)
    print(f"{'N':>6}  {'Err (2nd)':>12}  {'Err (4th)':>12}  {'Speedup':>10}")
    print("-" * 60)
    for i, N in enumerate(N_values):
        speedup = errors_2nd[i] / errors_4th[i] if errors_4th[i] > 0 else float('inf')
        print(f"{N:>6}  {errors_2nd[i]:>12.2e}  {errors_4th[i]:>12.2e}  {speedup:>10.1f}×")

    fig, ax = plt.subplots(figsize=(8, 5))
    N_arr = np.array(N_values, dtype=float)
    ax.loglog(N_values, errors_2nd, 'o-', label=r'2nd order: $O(\Delta x^2)$')
    ax.loglog(N_values, errors_4th, 's-', label=r'4th order: $O(\Delta x^4)$')
    ax.loglog(N_arr, errors_2nd[0] * (N_arr[0]/N_arr)**2, '--', color='gray',
              alpha=0.5, label='$N^{-2}$ reference')
    ax.loglog(N_arr, errors_4th[0] * (N_arr[0]/N_arr)**4, ':', color='gray',
              alpha=0.5, label='$N^{-4}$ reference')
    ax.set_xlabel('Number of grid points N')
    ax.set_ylabel('Relative error in $E_1$')
    ax.set_title('Convergence: 2nd vs 4th order finite differences')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\n4th-order stencil: (-1, 16, -30, 16, -1) / (12 Δx²)")
    print("The steeper slope on the log-log plot confirms O(Δx⁴) convergence.")

exercise_5_reference()